<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/llm/nebius_serverless_endpoint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG with a Nebius Serverless Endpoint

Use `OpenAILike` to query a vLLM model deployed on a [Nebius Serverless Endpoint](https://docs.nebius.com/serverless/overview). This example embeds three short documents locally on the CPU, retrieves relevant context, and sends it to your Endpoint for generation.

Serverless Endpoints run user-selected containers. They are separate from Nebius Token Factory managed model APIs, which have their own LlamaIndex integrations. This notebook uses an existing Endpoint; it does not create infrastructure or submit Serverless Jobs.

## Deploy the inference model first

Follow the [vLLM Nebius deployment guide](https://docs.vllm.ai/en/latest/deployment/frameworks/nebius/), including its readiness and authentication checks. This notebook targets the guide's [pinned single-GPU configuration](https://github.com/vllm-project/vllm/blob/5c642796e871e41ae75be0b402a8eaa44ee771b1/docs/deployment/frameworks/nebius.md):

- vLLM `0.19.1`, `Qwen/Qwen3-0.6B` at revision `c1899de289a04d12100db370d81485cdf75e47ca`;
- `--max-model-len 4096`, one HTTP port on 8000, and Endpoint token authentication;
- the guide's pinned Linux amd64 image and model/tokenizer revisions.

Save the Endpoint ID for cleanup. Copy its managed HTTPS **root URL** and Endpoint token below. The token is separate from your Nebius CLI credentials and Token Factory API key. Do not put it in notebook source or saved outputs. The Endpoint continues running until stopped or deleted, including when this notebook fails.

## Install the client dependencies

Use a fresh Python 3.11 environment or CPU Colab runtime. Restart the kernel after installation if these libraries were already imported. The versions below pin the client and embedding stack; the GPU server runs independently. Model/tokenizer files are downloaded from Hugging Face on first use.

In [ ]:
%pip install \
    "llama-index-core==0.14.24" \
    "llama-index-llms-openai-like==0.8.0" \
    "llama-index-llms-openai==0.7.10" \
    "llama-index-embeddings-huggingface==0.8.0" \
    "openai==2.54.0" \
    "httpx==0.28.1" \
    "sentence-transformers==5.7.0" \
    "transformers==4.57.6" \
    "huggingface-hub==0.36.2" \
    "torch==2.14.0"


## Connect to the Endpoint

Set `NEBIUS_ENDPOINT_URL` and `NEBIUS_ENDPOINT_TOKEN` in your environment, or use the prompts. Supply the root URL without `/v1`. Readiness requires both a healthy server and the expected served model ID; a control-plane `RUNNING` state alone is insufficient. Rerun this cell after starting a stopped Endpoint and retrieving its current URL.

In [ ]:
import getpass
import os
from urllib.parse import urlparse

import httpx

endpoint_url = (
    (
        os.environ.get("NEBIUS_ENDPOINT_URL")
        or input("Endpoint HTTPS root URL: ")
    )
    .strip()
    .rstrip("/")
)
parsed = urlparse(endpoint_url)
if (
    parsed.scheme != "https"
    or not parsed.hostname
    or parsed.username is not None
    or parsed.password is not None
    or parsed.path
    or parsed.query
    or parsed.fragment
):
    raise ValueError("Supply the managed HTTPS root URL, without /v1")
endpoint_token = os.environ.get("NEBIUS_ENDPOINT_TOKEN") or getpass.getpass(
    "Endpoint token: "
)
if not endpoint_token.strip():
    raise ValueError("An Endpoint token is required")

MODEL_ID = "Qwen/Qwen3-0.6B"
MODEL_REVISION = "c1899de289a04d12100db370d81485cdf75e47ca"
with httpx.Client(
    headers={"Authorization": f"Bearer {endpoint_token}"}, timeout=10.0
) as client:
    client.get(f"{endpoint_url}/health").raise_for_status()
    models_response = client.get(f"{endpoint_url}/v1/models")
    models_response.raise_for_status()
    if MODEL_ID not in {item["id"] for item in models_response.json()["data"]}:
        raise ValueError(f"The Endpoint does not serve {MODEL_ID}")
print("The Endpoint is ready and serves the expected model.")

## Configure generation explicitly

`context_window` describes the server's configured limit; it does not change that limit. Reserve room for output and chat-template overhead. Here, `is_chat_model=True` makes both `chat()` and `complete()` use Chat Completions. For a completion-only server, use `False` and validate the model's prompt formatting separately.

Function calling and structured outputs are disabled. Setting a capability flag does not enable server support. The Qwen-specific `enable_thinking` option keeps this small example's output short; remove or adapt it for other models. Automatic request retries are disabled so a failure does not silently repeat generation.

In [ ]:
from transformers import AutoTokenizer

from llama_index.core import Settings
from llama_index.llms.openai_like import OpenAILike

llm_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, trust_remote_code=False
)
Settings.tokenizer = llm_tokenizer.encode
llm = OpenAILike(
    model=MODEL_ID,
    api_base=f"{endpoint_url}/v1",
    api_key=endpoint_token,
    context_window=4096,
    max_tokens=256,
    is_chat_model=True,
    is_function_calling_model=False,
    should_use_structured_outputs=False,
    tokenizer=llm_tokenizer,
    temperature=0,
    timeout=120.0,
    max_retries=0,
    additional_kwargs={
        "extra_body": {"chat_template_kwargs": {"enable_thinking": False}}
    },
)

## Configure embeddings separately

The generative Endpoint does not supply embeddings. Use a pinned `all-MiniLM-L6-v2` model on the CPU for this small English corpus. Passing `embed_model` explicitly prevents the index from selecting a default external embedding provider.

For larger documents, check the embedding tokenizer's input limit independently of the generation context window. A separate [Text Embeddings Inference service](https://developers.llamaindex.ai/python/framework-api-reference/embeddings/text_embeddings_inference/) is another option, but requires its own deployment, authentication and model/output validation.

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
embed_model = HuggingFaceEmbedding(
    model_name=EMBED_MODEL_ID,
    revision=EMBED_REVISION,
    device="cpu",
    normalize=True,
    embed_batch_size=8,
    trust_remote_code=False,
)

## Build and inspect retrieval

These synthetic documents make it easy to inspect the retrieved evidence. The local vector index is sufficient for this example; no external vector database is required.

In [ ]:
from llama_index.core import Document, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter

corpus = {
    "atlas-code": "The fictional Atlas project code is ORCHID.",
    "atlas-hours": "The fictional Atlas office opens at nine in the morning.",
    "atlas-owner": "The fictional Atlas help desk is owned by Mira.",
}
documents = [Document(id_=key, text=text) for key, text in corpus.items()]
splitter = SentenceSplitter(chunk_size=128, chunk_overlap=16)
index = VectorStoreIndex.from_documents(
    documents, embed_model=embed_model, transformations=[splitter]
)
question = "What is the Atlas project code?"
retrieved = index.as_retriever(similarity_top_k=2).retrieve(question)
assert any(node.node.ref_doc_id == "atlas-code" for node in retrieved)
for node in retrieved:
    print(node.node.ref_doc_id, node.node.get_content())

## Persist and reload locally

Persist the index on the notebook host, separately from the Endpoint's disposable container disk. A small manifest records the corpus and embedding configuration. Reload only trusted files using the same configuration; rebuild the index if the corpus, embedding model, normalization or chunking changes. This fresh directory is for the example, not a production backup or concurrent-writer scheme.

In [ ]:
import hashlib
import json
import tempfile
from pathlib import Path

from llama_index.core import StorageContext, load_index_from_storage

manifest = {
    "corpus_sha256": hashlib.sha256(
        json.dumps(corpus, sort_keys=True).encode()
    ).hexdigest(),
    "embedding_model": EMBED_MODEL_ID,
    "embedding_revision": EMBED_REVISION,
    "normalize": True,
    "chunk_size": 128,
    "chunk_overlap": 16,
}
persist_dir = Path(tempfile.mkdtemp(prefix="nebius-llamaindex-"))
index.storage_context.persist(persist_dir=str(persist_dir))
manifest_path = persist_dir / "example-manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

Reload the saved index. To reuse an earlier run, set `persist_dir` to its saved directory and reconstruct `manifest` from the expected configuration before running this cell.

In [ ]:
manifest_path = persist_dir / "example-manifest.json"
if json.loads(manifest_path.read_text(encoding="utf-8")) != manifest:
    raise ValueError("Index configuration changed; rebuild the index")
loaded_index = load_index_from_storage(
    StorageContext.from_defaults(persist_dir=str(persist_dir)),
    embed_model=embed_model,
)
print(f"Index saved to {persist_dir}")

## Query the remote model

The query engine embeds your question locally and sends retrieved text to the Endpoint. Inspect the source nodes along with the answer. The expected fixture answer is `ORCHID`; this small model is an integration example, not a model-quality benchmark. Source nodes identify retrieved evidence and do not guarantee the answer is correct.

In [ ]:
query_engine = loaded_index.as_query_engine(llm=llm, similarity_top_k=2)
answer = query_engine.query(question)
print(str(answer))
for source in answer.source_nodes:
    print(source.node.ref_doc_id, source.node.get_content())

### Stream another answer

This makes a second inference request. An interrupted stream may contain only a partial answer; do not automatically replay it.

In [ ]:
streaming_engine = loaded_index.as_query_engine(
    llm=llm, similarity_top_k=2, streaming=True
)
streaming_answer = streaming_engine.query("When does the Atlas office open?")
streaming_answer.print_response_stream()

## Failures and cleanup

- **401/403:** check the Endpoint token, not your CLI or Token Factory credentials.
- **404/502/503 or readiness timeout:** check Endpoint state, vLLM startup logs, port and model downloads. A timeout does not cancel provisioning; reconcile the existing Endpoint before creating another.
- **400 or context overflow:** check the served model, supported parameters and input/output token budget. Do not retry an unchanged invalid request.
- **Interrupted stream:** treat the answer as incomplete. Interrupting a client does not guarantee server computation stopped.

Use your saved Endpoint ID with the [Nebius stop/start and deletion instructions](https://docs.nebius.com/serverless/endpoints/manage#how-to-stop-or-start-an-endpoint). Wait for stop to complete before starting again; rediscover the URL and repeat readiness. To finish, delete the Endpoint and confirm that reading its ID returns `NotFound`. Separately mounted volumes have their own lifecycle. Closing this notebook does not stop the Endpoint or remove its charges. Keep the local index directory if you want to reuse it, or remove that exact directory when finished.

This example does not configure autoscaling, scale-to-zero, Serverless Jobs, tool calling or production RAG hosting. Validate your model's answers, token limits and actual network streaming before using it in an application.